In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from gymnasium import spaces
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.env_checker import check_env


# ==================== Custom Environment ====================
class Projectile1DEnv(gym.Env):
    """
    Environment for 1D Projectile Motion
    """    
    def __init__(self):
        super().__init__()
        
        # Physics parameters
        self.g = 9.81
        self.max_distance = 100.0
        self.max_angle = 89.0
        self.min_angle = 1.0
        self.max_force = 50.0
        self.min_force = 5.0
        self.tolerance = 0.1 # meters around target for success
        
        
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(1,), dtype=np.float32
        ) # Normalized target distance(input)
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(2,), dtype=np.float32
        ) # Normalized angle and force (output)
        
        self.target_distance = 0.0
    
    def reset(self, seed=None, options=None):
        """Reset environment with new random target"""
        super().reset(seed=seed)
        
        self.target_distance = np.random.uniform(10.0, self.max_distance)
        obs = np.array([self.target_distance / self.max_distance], dtype=np.float32)
        
        return obs, {}
    
    def calculate_distance(self, angle_deg, force):
        """Physics: R = (v² * sin(2θ)) / g"""
        angle_rad = np.deg2rad(angle_deg)
        return (force**2 * np.sin(2 * angle_rad)) / self.g
    
    def step(self, action):
        """Execute action and return reward"""
        # Denormalize action from [-1, 1] to actual ranges
        angle_norm, force_norm = action
        angle = self.min_angle + (angle_norm + 1) / 2 * (self.max_angle - self.min_angle)
        force = self.min_force + (force_norm + 1) / 2 * (self.max_force - self.min_force)
        
        angle = np.clip(angle, self.min_angle, self.max_angle)
        force = np.clip(force, self.min_force, self.max_force)
        
        # Calculate projectile distance
        actual_distance = self.calculate_distance(angle, force)
        error = abs(actual_distance - self.target_distance)
        
        # DENSE REWARD SHAPING - Critical fix!
        # Exponential reward provides gradient even when far from target
        distance_reward = np.exp(-error / 10.0) * 10.0
        
        # Tiered success bonuses
        if error < self.tolerance:
            success_bonus = 50.0  # Hit the target!
        elif error < 0.5:
            success_bonus = 20.0  # Very close
        elif error < 3.0:
            success_bonus = 5.0   # Close
        else:
            success_bonus = 0.0
        
        # Penalty for very poor performance
        penalty = -5.0 if error > 50.0 else 0.0
        
        reward = float(distance_reward + success_bonus + penalty)
        
        # Episode ends after one shot (single-step MDP)
        terminated = True
        truncated = False
        
        obs = np.array([self.target_distance / self.max_distance], dtype=np.float32)
        info = {
            'angle': float(angle),
            'force': float(force),
            'actual_distance': float(actual_distance),
            'target_distance': float(self.target_distance),
            'error': float(error),
            'success': bool(error < self.tolerance)
        }
        
        return obs, reward, terminated, truncated, info

In [ ]:
# ==================== Callback for Tracking ====================
class TrackingCallback(BaseCallback):
    """
    Callback to track training metrics
    """
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self.episode_errors = []
        self.episode_successes = []
        self.episode_count = 0
    
    def _on_step(self) -> bool:
        # Get info from environment
        for info in self.locals.get('infos', []):
            if 'error' in info:
                self.episode_rewards.append(self.locals['rewards'][0])
                self.episode_errors.append(info['error'])
                self.episode_successes.append(1 if info['success'] else 0)
                self.episode_count += 1
                
                # Print progress every 500 episodes
                if self.episode_count % 500 == 0:
                    avg_reward = np.mean(self.episode_rewards[-500:])
                    avg_error = np.mean(self.episode_errors[-500:])
                    success_rate = np.mean(self.episode_successes[-500:]) * 100
                    print(f"Episode {self.episode_count:5d} | "
                          f"Reward: {avg_reward:7.2f} | "
                          f"Error: {avg_error:6.2f}m | "
                          f"Success: {success_rate:5.1f}%")
        
        return True


# ==================== Training Function ====================
def train_sac_projectile(total_timesteps=50000):
    """
    Train SAC agent on projectile environment
    """
    print("=" * 70)
    print("SAC FOR 1D PROJECTILE MOTION - STABLE-BASELINES3")
    print("=" * 70)
    print()
    
    # Create environment
    env = Projectile1DEnv()
    
    # Verify environment follows Gym interface
    print("Checking environment...")
    check_env(env, warn=True)
    print("✓ Environment check passed!")
    print()
    
    # Create SAC agent with optimized hyperparameters
    print("Creating SAC agent...")
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=3e-4,
        buffer_size=10000,
        learning_starts=1000,      # Start learning after 1000 steps
        batch_size=256,
        tau=0.005,                  # Soft update coefficient
        gamma=0.99,                 # Discount factor
        train_freq=1,               # Update after every step
        gradient_steps=1,
        ent_coef='auto',            # Automatic entropy tuning
        target_update_interval=1,
        verbose=0,
        seed=42
    )
    
    print("✓ SAC agent created")
    print("  Policy: MlpPolicy (2-layer 256-unit MLP)")
    print("  Learning rate: 3e-4")
    print("  Batch size: 256")
    print("  Buffer size: 100,000")
    print()
    
    # Create callback for tracking
    callback = TrackingCallback()
    
    # Train the agent
    print(f"Training for {total_timesteps:,} timesteps...")
    print("-" * 70)
    
    model.learn(
        total_timesteps=total_timesteps,
        callback=callback,
        progress_bar=False
    )
    
    print("-" * 70)
    print("Training completed!")
    print()
    
    # Save the model
    model.save("sac_projectile_model")
    print("✓ Model saved as 'sac_projectile_model.zip'")
    
    return model, callback, env


# ==================== Testing Function ====================
def test_agent(model, env, num_tests=100):
    """
    Test trained agent on random targets
    """
    print("\n" + "=" * 70)
    print("TESTING TRAINED AGENT")
    print("=" * 70)
    
    errors = []
    successes = 0
    results = []
    
    for _ in range(num_tests):
        obs, _ = env.reset()
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        
        errors.append(info['error'])
        if info['success']:
            successes += 1
        results.append(info)
    
    # Print statistics
    mean_error = np.mean(errors)
    std_error = np.std(errors)
    success_rate = successes / num_tests * 100
    
    print(f"\nTest Results ({num_tests} trials):")
    print(f"  Mean Error: {mean_error:.2f} ± {std_error:.2f}m")
    print(f"  Success Rate: {success_rate:.1f}%")
    print(f"  Min Error: {np.min(errors):.2f}m")
    print(f"  Max Error: {np.max(errors):.2f}m")
    
    # Show some examples
    print("\n" + "-" * 70)
    print("Example Predictions:")
    print(f"{'Target (m)':>12} | {'Angle (°)':>10} | {'Force (m/s)':>12} | "
          f"{'Actual (m)':>11} | {'Error (m)':>10}")
    print("-" * 70)
    
    test_distances = [15, 30, 50, 70, 90]
    for dist in test_distances:
        env.target_distance = dist
        obs = np.array([dist / env.max_distance], dtype=np.float32)
        action, _ = model.predict(obs, deterministic=True)
        _, _, _, _, info = env.step(action)
        
        print(f"{dist:>12.1f} | {info['angle']:>10.1f} | {info['force']:>12.2f} | "
              f"{info['actual_distance']:>11.2f} | {info['error']:>10.2f}")
    
    return {
        'mean_error': mean_error,
        'std_error': std_error,
        'success_rate': success_rate,
        'errors': errors,
        'results': results
    }


# ==================== Visualization ====================
def visualize_results(callback, test_results):
    """
    Create visualization of training and testing results
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Plot 1: Training Rewards
    ax = axes[0, 0]
    rewards = callback.episode_rewards
    window = 100
    smooth = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smooth, linewidth=2, color='blue')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title('Training Rewards (100-episode MA)')
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Training Errors
    ax = axes[0, 1]
    errors = callback.episode_errors
    smooth_err = np.convolve(errors, np.ones(window)/window, mode='valid')
    ax.plot(smooth_err, linewidth=2, color='red')
    ax.axhline(y=2.0, color='green', linestyle='--', label='Success Threshold')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Error (m)')
    ax.set_title('Training Errors (100-episode MA)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Success Rate
    ax = axes[0, 2]
    successes = callback.episode_successes
    success_smooth = np.convolve(successes, np.ones(window)/window, mode='valid') * 100
    ax.plot(success_smooth, linewidth=2, color='green')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Success Rate (%)')
    ax.set_title('Success Rate (100-episode MA)')
    ax.set_ylim([0, 105])
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Test Error Distribution
    ax = axes[1, 0]
    ax.hist(test_results['errors'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
    ax.axvline(x=2.0, color='red', linestyle='--', linewidth=2, label='Success Threshold')
    ax.set_xlabel('Error (m)')
    ax.set_ylabel('Frequency')
    ax.set_title('Test Error Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 5: Action Distribution
    ax = axes[1, 1]
    angles = [r['angle'] for r in test_results['results']]
    forces = [r['force'] for r in test_results['results']]
    scatter = ax.scatter(angles, forces, c=test_results['errors'], 
                        cmap='RdYlGn_r', s=100, alpha=0.6, edgecolors='black')
    ax.set_xlabel('Angle (°)')
    ax.set_ylabel('Force (m/s)')
    ax.set_title('Learned Actions')
    plt.colorbar(scatter, ax=ax, label='Error (m)')
    ax.grid(True, alpha=0.3)
    
    # Plot 6: Performance Metrics
    ax = axes[1, 2]
    metrics = ['Mean\nError (m)', 'Std\nError (m)', 'Success\nRate (%)']
    values = [test_results['mean_error'], test_results['std_error'], 
              test_results['success_rate']]
    colors = ['coral', 'skyblue', 'lightgreen']
    bars = ax.bar(metrics, values, color=colors, edgecolor='black', alpha=0.7)
    ax.set_ylabel('Value')
    ax.set_title('Test Metrics')
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('sac_projectile_results_sb3.png', dpi=150, bbox_inches='tight')
    print("\n✓ Visualization saved as 'sac_projectile_results_sb3.png'")
    plt.show()

In [ ]:
# ==================== Main Execution ====================

# Train the agent
model, callback, env = train_sac_projectile(total_timesteps=100000)

In [ ]:
# Test the agent
test_results = test_agent(model, env, num_tests=1000)

# Visualize results
visualize_results(callback, test_results)

In [ ]:
env = Projectile1DEnv()

model = SAC.load("sac_projectile_model", env=env)
# Test the agent
test_results = test_agent(model, env, num_tests=100000)

print("\n" + "=" * 70)
print("✓ COMPLETE!")
print(f"✓ Final Success Rate: {test_results['success_rate']:.1f}%")
print(f"✓ Final Mean Error: {test_results['mean_error']:.2f}m")
print("=" * 70)

In [ ]:
env.target_distance=90
obs1=np.array([env.target_distance/env.max_distance],dtype=np.float32)
# obs, _ = env.reset()

In [ ]:
action, _ = model.predict(obs1, deterministic=True)

In [ ]:
action_, _, _, _, info = env.step(action)
        
info

In [ ]:
def plot_current_trajectory(model, env):
    """
    Plot trajectory for the current target in env
    Uses whatever target is already set in env.target_distance
    """
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Use current target distance
    target_dist = env.target_distance
    obs = np.array([target_dist / env.max_distance], dtype=np.float32)
    
    # Get model's prediction
    action, _ = model.predict(obs, deterministic=True)
    _, _, _, _, info = env.step(action)
    
    angle = info['angle']
    force = info['force']
    actual_dist = info['actual_distance']
    error = info['error']
    
    # Calculate trajectory
    angle_rad = np.deg2rad(angle)
    t_flight = (2 * force * np.sin(angle_rad)) / env.g
    t_points = np.linspace(0, t_flight, 100)
    
    x = force * np.cos(angle_rad) * t_points
    y = force * np.sin(angle_rad) * t_points - 0.5 * env.g * t_points**2
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot trajectory
    ax.plot(x, y, 'b-', linewidth=3, label='Trajectory')
    
    # Launch point
    ax.plot(0, 0, 'go', markersize=15, label='Launch Point', zorder=5)
    
    # Landing point
    ax.plot(actual_dist, 0, 'ro', markersize=12, 
            label=f'Landing: {actual_dist:.2f}m', zorder=5)
    
    # Target zone
    target_zone = plt.Rectangle(
        (target_dist - env.tolerance, -1),
        2 * env.tolerance, 
        1,
        color='green', alpha=0.3, label=f'Target Zone'
    )
    ax.add_patch(target_zone)
    ax.axvline(x=target_dist, color='green', linestyle='--', 
               linewidth=2, alpha=0.6, label=f'Target: {target_dist:.2f}m')
    
    # Apex
    # apex_idx = np.argmax(y)
    # ax.plot(x[apex_idx], y[apex_idx], 'y*', markersize=20, 
    #         label=f'Apex: {y[apex_idx]:.2f}m', zorder=5)
    
    # Ground
    ax.axhline(y=0, color='brown', linewidth=3, alpha=0.7)
    
    # Title
    success = "✓ HIT!" if error < env.tolerance else "✗ MISS"
    color = 'green' if error < env.tolerance else 'red'
    ax.set_title(
        f'{success} | Target: {target_dist:.2f}m | Error: {error:.2f}m\n'
        f'Launch Angle: {angle:.2f}° | Launch Velocity: {force:.2f}m/s',
        fontsize=14, fontweight='bold', color=color, pad=20
    )
    
    # Labels
    ax.set_xlabel('Horizontal Distance (m)', fontsize=12)
    ax.set_ylabel('Height (m)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11, loc='upper right')
    
    # Limits
    max_x = max(target_dist * 1.1, actual_dist * 1.1)
    ax.set_xlim(-5, max_x)
    ax.set_ylim(-2, np.max(y) * 1.2)
    
    plt.tight_layout()
    plt.savefig('current_trajectory.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Trajectory plot saved: Target={target_dist:.2f}m, Error={error:.2f}m")
    plt.show()

In [ ]:
plot_current_trajectory(model, env)